# BERT + LoRA: comparing author styles

Fine-tune a shared BERT encoder on **same-author / different-author** paper pairs from `2_Data/data_reader.ipynb`. Only titles and abstracts enter the model; first-author ORCIDs supply labels.

Run the cells in order. Install `3_train/requirements.txt` in the notebook's Python environment first. Defaults are a small smoke run; CUDA uses mixed precision on one GPU. This learns authorship similarity, which may include topic as well as style; it is not a topic-controlled style benchmark.

## 1. Imports
Keep dependencies here and experiment settings in the next section.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import (
    LoraConfig, TaskType, get_peft_model,
    get_peft_model_state_dict, set_peft_model_state_dict,
)
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, set_seed


/home/petrecc1/miniforge3/envs/SciARvenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration
Defaults use a small sample of `sciar_refereed_v0` (including single-paper authors). Pair counts are sampled examples, not all possible pairs. Set `MAX_PAPERS = None` and increase pair counts and epochs for a larger run.

In [2]:
# Works when Jupyter starts from the repository root or 3_train/.
ROOT = Path.cwd().resolve()
if ROOT.name == '3_train':
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/processed/sciar_refereed_v0/papers.parquet'
READER_PATH = ROOT / '2_Data/data_reader.ipynb'
OUTPUT_DIR = ROOT / '3_train/outputs/bert_lora_smoke'

MODEL_NAME = 'bert-base-uncased'
SEED = 42
MAX_PAPERS = 10_000  # Read one small Parquet batch for a smoke run.
TRAIN_PAIRS = 64
VAL_PAIRS = 32
TEST_PAIRS = 32
MAX_LENGTH = 128  # Per paper, including title and abstract.
BATCH_SIZE = 8  # Each pair contains two independently encoded papers.
EPOCHS = 1
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MARGIN = 0.2
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
USE_AMP = USE_CUDA
assert DATA_PATH.is_file(), f'Dataset not found: {DATA_PATH}'
assert READER_PATH.is_file(), f'Reader not found: {READER_PATH}'
assert all(n >= 2 and n % 2 == 0 for n in (TRAIN_PAIRS, VAL_PAIRS, TEST_PAIRS))
assert EPOCHS >= 1 and BATCH_SIZE >= 1 and 8 <= MAX_LENGTH <= 512
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(f'Device: {DEVICE} | Mixed precision: {USE_AMP} | Output: {OUTPUT_DIR}')


Device: cpu | Mixed precision: False | Output: /home/petrecc1/projects/SciAR/3_train/outputs/bert_lora_smoke


/home/petrecc1/miniforge3/envs/SciARvenv/lib/python3.14/site-packages/torch/cuda/__init__.py:241: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 3. Load papers and split authors
Use the reader's 80/10/10 author split before generating pairs. Single-paper authors can contribute negative pairs. Load only the tagged reader cells so its demo does not run. Save author assignments for reproducibility.

In [3]:
reader_cells = json.loads(READER_PATH.read_text())['cells']
reader = {}
for cell in reader_cells:
    if cell['cell_type'] == 'code' and 'reader-code' in cell.get('metadata', {}).get('tags', []):
        exec(''.join(cell['source']), reader)
load_papers = reader['load_papers']
split_authors = reader['split_authors']
make_pairs = reader['make_pairs']

papers = load_papers(DATA_PATH, max_papers=MAX_PAPERS)
assert papers['bibcode'].is_unique, 'Expected one row per bibcode.'
author_splits = split_authors(papers, seed=SEED)
assert all(len(ids) >= 2 for ids in author_splits.values()), 'Need at least two authors per split.'
assert not (set(author_splits['train']) & set(author_splits['validation'])
            or set(author_splits['train']) & set(author_splits['test'])
            or set(author_splits['validation']) & set(author_splits['test']))
paper_groups = papers.groupby('first_author_orcid').indices
(OUTPUT_DIR / 'author_splits.json').write_text(json.dumps(author_splits, indent=2))
for name, ids in author_splits.items():
    print(f'{name}: {len(ids):,} authors, {sum(len(paper_groups[a]) for a in ids):,} papers')


train: 4,758 authors, 7,931 papers
validation: 594 authors, 1,017 papers
test: 596 authors, 1,052 papers


## 4. Sample balanced pairs
The reader makes balanced same-author (`1`) and different-author (`0`) pairs. Pairs can recur within a split. Save bibcodes and 0/1 labels for tracking; the cosine loss converts 0 to -1 when training.

In [4]:
pair_sets = {
    name: make_pairs(papers, paper_groups, author_splits[name], size, SEED + offset)
    for offset, (name, size) in enumerate(
        [('train', TRAIN_PAIRS), ('validation', VAL_PAIRS), ('test', TEST_PAIRS)]
    )
}
for name, examples in pair_sets.items():
    assert examples['label'].value_counts().to_dict() == {0: len(examples) // 2, 1: len(examples) // 2}
    examples[['bibcode_a', 'bibcode_b', 'label']].to_csv(OUTPUT_DIR / f'{name}_pairs.csv', index=False)
    print(f'{name}: {len(examples):,} pairs')


train: 64 pairs
validation: 32 pairs
test: 32 pairs


## 5. Tokenizer and LoRA encoder
Tokenize each reader text (title, blank line, abstract) on its own. Both papers share the same encoder. Only LoRA query/value adapters train; the base BERT weights stay frozen. Masked mean pooling produces a normalized embedding without using BERT's pooler.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False)
model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['query', 'value'], bias='none',
)).to(DEVICE)
model.print_trainable_parameters()

def collate_pairs(batch):
    # first half is A, second half is B; encode each paper separately
    texts = [row['text_a'] for row in batch] + [row['text_b'] for row in batch]
    tokens = tokenizer(
        texts,
        padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt',
    )
    # CosineEmbeddingLoss expects -1 for different authors, while saved labels stay 0/1
    labels = torch.tensor([1 if row['label'] == 1 else -1 for row in batch], dtype=torch.float32)
    return tokens, labels

loaders = {
    name: DataLoader(
        examples.to_dict('records'), batch_size=BATCH_SIZE, shuffle=(name == 'train'),
        collate_fn=collate_pairs, num_workers=0, pin_memory=USE_CUDA,
        generator=torch.Generator().manual_seed(SEED),
    )
    for name, examples in pair_sets.items()
}

def encode(tokens):
    tokens = {key: value.to(DEVICE, non_blocking=USE_CUDA) for key, value in tokens.items()}
    hidden = model(**tokens).last_hidden_state
    mask = tokens['attention_mask'].unsqueeze(-1).to(hidden.dtype)
    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return F.normalize(pooled, p=2, dim=-1)


trainable params: 294,912 || all params: 109,186,560 || trainable%: 0.2701


## 6. Loss and evaluation
Use `CosineEmbeddingLoss(margin=0.2)`: **same author:** `1 − cosine_similarity`; **different author:** `max(0, cosine_similarity − margin)`. CUDA forward passes use float16 autocast and gradient scaling; the loss uses float32 embeddings. Validation ROC-AUC selects the best epoch; a validation-only threshold maximizes balanced accuracy (equivalent to accuracy for these balanced pairs).

In [6]:
criterion = torch.nn.CosineEmbeddingLoss(margin=MARGIN)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    for tokens, labels in tqdm(loader, desc='Evaluate', leave=False):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            left, right = encode(tokens).chunk(2)
        loss = criterion(left.float(), right.float(), labels.to(DEVICE, non_blocking=USE_CUDA))
        total_loss += loss.item() * len(labels)
        all_labels.extend((labels > 0).int().tolist())
        all_scores.extend((left.float() * right.float()).sum(dim=-1).cpu().tolist())
    return total_loss / len(loader.dataset), np.array(all_labels), np.array(all_scores)


/tmp/ipykernel_1902040/1628241315.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## 7. Train
Record an untrained validation baseline, then optimize the LoRA parameters. Keep the best adapter in memory according to validation ROC-AUC. CUDA uses mixed precision; CPU runs in full precision.

In [7]:
baseline_loss, baseline_labels, baseline_scores = evaluate(loaders['validation'])
baseline_auc = float(roc_auc_score(baseline_labels, baseline_scores))
print(f'Before training: validation loss={baseline_loss:.4f}, AUC={baseline_auc:.4f}')
history, best_auc, best_state = [], -float('inf'), None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for tokens, labels in tqdm(loaders['train'], desc=f'Epoch {epoch}/{EPOCHS}'):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            left, right = encode(tokens).chunk(2)
        loss = criterion(left.float(), right.float(), labels.to(DEVICE, non_blocking=USE_CUDA))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(labels)
    val_loss, val_labels, val_scores = evaluate(loaders['validation'])
    val_auc = float(roc_auc_score(val_labels, val_scores))
    row = dict(epoch=epoch, train_loss=total_loss / TRAIN_PAIRS, val_loss=val_loss, val_auc=val_auc)
    history.append(row)
    print(row)
    if val_auc > best_auc:
        best_auc, best_epoch = val_auc, epoch
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in get_peft_model_state_dict(model).items()
        }
set_peft_model_state_dict(model, best_state)
pd.DataFrame(history)


Before training: validation loss=0.3440, AUC=0.9844


Epoch 1/1: 100%|██████████| 8/8 [00:21<00:00,  2.74s/it]
                                                       

{'epoch': 1, 'train_loss': 0.34313466027379036, 'val_loss': 0.3345825970172882, 'val_auc': 0.984375}


,epoch,train_loss,val_loss,val_auc
0,1,0.343135,0.334583,0.984375


## 8. Held-out evaluation and export
Choose the decision threshold using the best adapter's validation scores, then evaluate the test set once. Save the adapter, tokenizer, metrics, configuration, history, and sampling manifests. The saved model needs the original base BERT checkpoint; it is not a standalone merged model.

In [8]:
val_loss, val_labels, val_scores = evaluate(loaders['validation'])
fpr, tpr, thresholds = roc_curve(val_labels, val_scores)
finite = np.isfinite(thresholds)
threshold = float(thresholds[finite][np.argmax((tpr - fpr)[finite])])
test_loss, test_labels, test_scores = evaluate(loaders['test'])
metrics = {
    'baseline_val_auc': baseline_auc, 'best_epoch': best_epoch,
    'val_auc': float(roc_auc_score(val_labels, val_scores)),
    'val_accuracy': float(accuracy_score(val_labels, val_scores >= threshold)),
    'threshold': threshold, 'test_loss': test_loss,
    'test_auc': float(roc_auc_score(test_labels, test_scores)),
    'test_accuracy': float(accuracy_score(test_labels, test_scores >= threshold)),
}
model.save_pretrained(OUTPUT_DIR / 'adapter')
tokenizer.save_pretrained(OUTPUT_DIR / 'adapter')
config = {key: str(value) if isinstance(value, Path) else value
          for key, value in list(globals().items())
          if key.isupper() and isinstance(value, (str, int, float, Path))}
config.update(pooling='attention-mask mean, including special tokens', normalize_embeddings=True)
(OUTPUT_DIR / 'config.json').write_text(json.dumps(config, indent=2))
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'history.csv', index=False)
print(json.dumps(metrics, indent=2))
print(f'Saved to {OUTPUT_DIR}')


{
  "baseline_val_auc": 0.984375,
  "best_epoch": 1,
  "val_auc": 0.984375,
  "val_accuracy": 0.9375,
  "threshold": 0.829159140586853,
  "test_loss": 0.32287784665822983,
  "test_auc": 0.96875,
  "test_accuracy": 0.84375
}
Saved to /home/petrecc1/projects/SciAR/3_train/outputs/bert_lora_smoke
